In [1]:
# 1a
class GoalBasedAgent:
    def __init__(self, limit: int):
        self.limit: int = limit
        print(f"Agent Created with Depth Limit: {self.limit}.")

    def dls(self, graph, node, goal, depth=0, visited=None):
        if visited is None:
            visited = set()

        visited.add(node)
        print(f"Visiting: {node}")
        if node == goal:
            return True

        if depth == self.limit:
            print("Depth Limit Reached Returning!")
            return False

        for neighbor in graph[node]:
            if self.dls(graph, neighbor, goal, depth + 1, visited):
                return True
        return False


def run_agent(agent: GoalBasedAgent, graph: dict, start_node: str, goal: str):
    if agent.dls(graph, start_node, goal):
        print(f"Found Node: {goal}!")
        return
    print("Node Not Found!")


graph = {
    "A": ["B", "C"],
    "B": ["D", "E", "A"],  # loop
    "C": ["F", "G"],
    "D": ["H"],
    "E": ["A"],
    "F": ["I"],
    "G": [],
    "H": [],
    "I": [],
}

gba = GoalBasedAgent(2)

start_node = "A"
goal = "G"
print(f"Goal: {goal}")
run_agent(gba, graph, start_node, goal)

print()

start_node = "A"
goal = "I"
print(f"Goal: {goal}")
run_agent(gba, graph, start_node, goal)

Agent Created with Depth Limit: 2.
Goal: G
Visiting: A
Visiting: B
Visiting: D
Depth Limit Reached Returning!
Visiting: E
Depth Limit Reached Returning!
Visiting: A
Depth Limit Reached Returning!
Visiting: C
Visiting: F
Depth Limit Reached Returning!
Visiting: G
Found Node: G!

Goal: I
Visiting: A
Visiting: B
Visiting: D
Depth Limit Reached Returning!
Visiting: E
Depth Limit Reached Returning!
Visiting: A
Depth Limit Reached Returning!
Visiting: C
Visiting: F
Depth Limit Reached Returning!
Visiting: G
Depth Limit Reached Returning!
Node Not Found!


In [2]:
# 1b
class UtilBasedAgent:
    def __init__(self, goal):
        self.goal = goal

    def utility(self, cost):
        return -cost

    def select_best(self, states):
        states.sort(key=lambda x: self.utility(x[0]), reverse=True)
        return states.pop(0)

    def ucs(self, graph, start):
        pq = [(0, start, [start])]
        visited = set()

        while pq:
            pq.sort(key=lambda x: x[0])
            cost, node, path = self.select_best(pq)

            if node in visited:
                continue

            visited.add(node)
            print(f"visiting: {node}, Total Cost: {cost}")

            if node == self.goal:
                return path, cost

            for neighbor, edgecost in graph.get(node, []):
                if neighbor not in visited:
                    total_cost = cost + edgecost
                    pq.append((total_cost, neighbor, path + [neighbor]))

        return None, float("inf")


def run_agent(agent, graph, start):
    path, cost = agent.ucs(graph, start)
    print(f"Path: {path}\nTotal Cost: {cost}")


graph = {
    "A": [("B", 1), ("C", 4)],
    "B": [("D", 2), ("E", 5)],
    "C": [("F", 1)],
    "D": [],
    "E": [],
    "F": [],
}

uba = UtilBasedAgent("F")
run_agent(uba, graph, "A")

visiting: A, Total Cost: 0
visiting: B, Total Cost: 1
visiting: D, Total Cost: 3
visiting: C, Total Cost: 4
visiting: F, Total Cost: 5
Path: ['A', 'C', 'F']
Total Cost: 5


In [3]:
# 2
graph = {
    "1": [("2", 10), ("3", 15), ("4", 20)],
    "2": [("1", 10), ("3", 35), ("4", 25)],
    "3": [("1", 15), ("2", 35), ("4", 30)],
    "4": [("1", 20), ("2", 25), ("3", 30)],
}


def travelling_salesman(graph, start):
    # (cost, curr, visited_set, path)
    pq = [(0, start, {start}, [start])]
    n = len(graph)

    while pq:
        pq.sort(key=lambda x: x[0])
        cost, node, visited, path = pq.pop(0)

        if len(visited) == n:
            for neighbor, edgecost in graph[node]:
                if start == neighbor:
                    return cost + edgecost, path + [start]

        for neighbor, edgecost in graph[node]:
            if neighbor not in visited:
                new_cost = cost + edgecost
                new_visited = visited | {neighbor}
                pq.append((new_cost, neighbor, new_visited, path + [neighbor]))

    return None, None


start = "1"
cost, path = travelling_salesman(graph, start)
print(f"Cost: {cost}\nPath: {path}")

Cost: 80
Path: ['1', '2', '4', '3', '1']


In [4]:
# 3
def dls(graph, node, goal, depth, path):
    if depth == 0:
        return False
    if node == goal:
        path.append(node)
        return True
    if node not in graph:
        return False
    for child in graph[node]:
        if dls(graph, child, goal, depth - 1, path):
            path.append(node)
            return True
    return False


def iterative_deepening(graph, start, goal, max_depth=10):
    for depth in range(max_depth + 1):
        print(f"Depth: {depth}")
        path = []
        if dls(graph, start, goal, depth, path):
            print("\nPath to goal:", "->".join(reversed(path)))
            return
    print("Goal not found within depth limit.")


tree = {
    "A": ["B", "C"],
    "B": ["D", "E"],
    "C": ["F", "G"],
    "D": ["H"],
    "E": [],
    "F": ["I"],
    "G": [],
    "H": [],
    "I": [],
}

graph = {
    "Corridor A": ["Storage", "Station", "Room 101", "Room 102"],
    "Storage": ["Corridor A"],
    "Station": ["Corridor A", "Corridor B"],
    "Room 101": ["Corridor A"],
    "Room 102": ["Corridor A"],
    "Corridor B": ["Storage", "Station", "Room 201", "Room 202"],
    "Room 201": ["Corridor B"],
    "Room 202": ["Corridor B"],
}


print("Iterative Deepening on Tree: ")
iterative_deepening(tree, "A", "I")

print("\nIterative Deepening on Graph: ")
iterative_deepening(graph, "Corridor A", "Room 202")

Iterative Deepening on Tree: 
Depth: 0
Depth: 1
Depth: 2
Depth: 3
Depth: 4

Path to goal: A->C->F->I

Iterative Deepening on Graph: 
Depth: 0
Depth: 1
Depth: 2
Depth: 3
Depth: 4

Path to goal: Corridor A->Station->Corridor B->Room 202
